# LangChain: Memory

## Outline
* ConversationBufferMemory
* ConversationBufferWindowMemory
* ConversationTokenBufferMemory
* ConversationSummaryMemory

## ConversationBufferMemory

In [22]:
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

import warnings
warnings.filterwarnings('ignore')


In [23]:
from IPython.display import display, HTML

display(HTML("""
<style>
.output_area pre {
    white-space: pre-wrap !important;
    word-wrap: break-word !important;
}
</style>
"""))

Note: LLM's do not always produce the same results. When executing the code in your notebook, you may get slightly different answers that those in the video.

In [24]:
llm_model = "google/gemma-4-26b-a4b-it:free"

In [25]:
from langchain_openai import ChatOpenAI
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory


In [26]:
llm = ChatOpenAI(temperature=0.0, 
                model=llm_model,
                api_key=os.environ["OPENROUTER_API_KEY"],
                base_url="https://openrouter.ai/api/v1"
)
memory = InMemoryChatMessageHistory()

store = {}

def get_session_history(session_id):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

conversation = RunnableWithMessageHistory(
    llm,
    get_session_history
)    

In [27]:
response = conversation.invoke(
    "Hi, my name is Andrew",
    config={"configurable":{"session_id":"andrew"}}
)

In [28]:
response = conversation.invoke(
    "What is 1+1?",
    config={"configurable":{"session_id":"andrew"}}
)

In [29]:
response = conversation.invoke(
    "What is My Name",
    config={"configurable":{"session_id":"andrew"}}
)

In [30]:
memory = get_session_history("andrew")

In [37]:
for message in memory.messages:
    print(message.content)

Hi, my name is Andrew
Hello, Andrew! It's nice to meet you. How can I help you today?
What is 1+1?
1 + 1 = 2
What is My Name
Your name is Andrew.


In [ ]:
memory = ConversationBufferMemory() #This is old langchain

In [44]:
#this is the new langchain method
memory = InMemoryChatMessageHistory();

In [45]:
'''memory.save_context({"input": "Hi"}, 
                    {"output": "What's up"})  these are old''' 

memory.add_user_message("Hi")
memory.add_ai_message("What's up")

In [46]:
for message in memory.messages:
    print(message.content)

Hi
What's up


In [ ]:
memory.load_memory_variables({}) #this is old and not needed; previous loop is enough

In [47]:
#this is the new langchain methods ; combining the previous
memory = InMemoryChatMessageHistory();

memory.add_user_message("Hi")
memory.add_ai_message("What's up")

for message in memory.messages:
    print(message.content)

Hi
What's up


In [48]:
memory.add_user_message("Not much, just hanging")
memory.add_ai_message("Cool")


In [50]:
for message in memory.messages:
    print(message.content)

Hi
What's up
Not much, just hanging
Cool


## ConversationBufferWindowMemory

In [51]:
from langchain_core.chat_history import InMemoryChatMessageHistory

In [ ]:
memory = ConversationBufferWindowMemory(k=1)     #old to see the latest 1 conversation          

In [52]:
#this is the new way
memory = InMemoryChatMessageHistory()

In [54]:
memory.add_user_message( "Hi")
memory.add_ai_message("What's up")
memory.add_user_message( "Not much, just hanging")
memory.add_ai_message("Cool")

recent_message = memory.messages[-2:]
for message in recent_message:
    print(message.content)


Not much, just hanging
Cool


In [ ]:
memory.load_memory_variables({}) #not needed previous for loop did the job

In [61]:
llm = ChatOpenAI(
    temperature=0.0,
    model="google/gemma-4-26b-a4b-it:free",
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url="https://openrouter.ai/api/v1"
)

store = {}

def get_session_history(session_id):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    history = store[session_id]

    history.messages = history.messages[-2:]

    #only returning the last having convo
    return history    

conversation = RunnableWithMessageHistory(
    llm,
    get_session_history
)

In [67]:
response = conversation.invoke(
    "Hi, my name is Andrew",
    config={"configurable":{"session_id":"andrew"}}
)
print(response.content)

It's nice to meet you, Andrew! How can I help you today?


In [68]:
memory.add_user_message("What is 1+1?")
response = conversation.invoke(
    "What is 1+1",
    config={"configurable":{"session_id":"andrew"}}
)
print(response.content)

1 + 1 = 2


In [69]:
response = conversation.invoke(
    "What is my name?",
    config={"configurable":{"session_id":"andrew"}}
)
print(response.content)

I do not know your name because I do not have access to your personal information or identity unless you have shared it with me in this conversation.


## ConversationTokenBufferMemory

In [70]:
!pip install tiktoken

In [74]:
import tiktoken

In [71]:
from langchain_core.messages import trim_messages

In [ ]:
memory = ConversationTokenBufferMemory(llm=llm, max_token_limit=50)
memory.save_context({"input": "AI is what?!"},
                    {"output": "Amazing!"})
memory.save_context({"input": "Backpropagation is what?"},
                    {"output": "Beautiful!"})
memory.save_context({"input": "Chatbots are what?"}, 
                    {"output": "Charming!"})

In [72]:
#this is the newest syntax

memory = InMemoryChatMessageHistory()

memory.add_user_message("AI is what?!")
memory.add_ai_message("Amazing!")

memory.add_user_message("Backpropagation is what?")
memory.add_ai_message("Beautiful!")

memory.add_user_message("Chatbots are what?")
memory.add_ai_message("Charming!")

In [ ]:
encoding = tiktoken.get_encoding("cl100k_base")

def token_counter(messages):
    return sum(
        len(encoding.encode(message.content))
        for message in messages
    ) #this method counts the number of tokens used so we can keep token counter

In [77]:
trimmed_messages = trim_messages(
    memory.messages,
    max_tokens=10,
    strategy="last",
    token_counter=token_counter,
    include_system=True,
    start_on="human",
    end_on=("human","tool")
)

for message in trimmed_messages:
    print(message.content)

Chatbots are what?


In [ ]:
memory.load_memory_variables({})

## ConversationSummaryMemory

In [ ]:
'''
This is very different from the summary memory shown in the tutorial, like really different
it uses langgraph and stuffs
'''

In [110]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver

In [111]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    temperature=0.0,
    model="google/gemma-4-26b-a4b-it:free",
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url="https://openrouter.ai/api/v1"
)

In [112]:
checkpointer = InMemorySaver()

In [113]:
agent = create_agent(
    model = llm,
    tools=[],
    middleware=[
        SummarizationMiddleware(
            model=llm,
            trigger=("tokens",100),
            keep=("messages",5)
        )
    ],
)

In [121]:

# Create a long string
schedule = """There is a meeting at 8am with your product team.
You will need your PowerPoint presentation prepared.
9am-12pm have time to work on your LangChain project,
which will go quickly because LangChain is such a powerful tool.
At Noon, lunch at the Italian restaurant with a customer
who is driving from over an hour away to meet you
to understand the latest in AI.
Be sure to bring your laptop to show the latest LLM demo."""


# Create a unique conversation thread
config = {
    "configurable": {
        "thread_id": "schedule-demo-2"
    }
}


# First conversation turn
response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Hello"
            }
        ]
    },
    config
)

print("USER: Hello")
print("AI:", response["messages"][-1].content)
print("-" * 60)


# Second conversation turn
response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Not much, just hanging"
            }
        ]
    },
    config
)

print("USER: Not much, just hanging")
print("AI:", response["messages"][-1].content)
print("-" * 60)


# Third conversation turn
response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": f"""What is on the schedule today?

{schedule}"""
            }
        ]
    },
    config
)

print("USER: What is on the schedule today?")
print("AI:", response["messages"][-1].content)
print("-" * 60)


USER: Hello
AI: Hello! How can I help you today?
------------------------------------------------------------
USER: Not much, just hanging
AI: Fair enough! Sometimes "just hanging" is the best way to be. 

Since you're just chilling, I'm here if you get bored. We could:

*   **Play a quick game** (Trivia, 20 Questions, or a word game).
*   **Find something to watch/read** (Give me a vibe, and I'll give you a recommendation).
*   **Learn something weird** (I can drop a random, useless fact on you).
*   **Or just keep it low-key.**

How's your day going otherwise? Low stress?
------------------------------------------------------------
USER: What is on the schedule today?
AI: Here is your schedule for today:

*   **8:00 AM:** Meeting with the Product Team (Note: Have your PowerPoint presentation prepared).
*   **9:00 AM – 12:00 PM:** Work on LangChain project.
*   **12:00 PM:** Lunch at the Italian restaurant with a customer (Note: Bring your laptop to show the latest LLM demo).
--------

In [114]:
for message in memory.messages:
    print(message.content)

Hello
What's up
Not much, just hanging
Cool
What is on the schedule today?
There is a meeting at 8am with your product team. You will need your powerpoint presentation prepared. 9am-12pm have time to work on your LangChain project which will go quickly because Langchain is such a powerful tool. At Noon, lunch at the italian resturant with a customer who is driving from over an hour away to meet you to understand the latest in AI. Be sure to bring your laptop to show the latest LLM demo.
Hello
What's up
Not much, just hanging
Cool
What is on the schedule today?
There is a meeting at 8am with your product team. You will need your powerpoint presentation prepared. 9am-12pm have time to work on your LangChain project which will go quickly because Langchain is such a powerful tool. At Noon, lunch at the italian resturant with a customer who is driving from over an hour away to meet you to understand the latest in AI. Be sure to bring your laptop to show the latest LLM demo.


In [ ]:
conversation = ConversationChain(
    llm=llm, 
    memory = memory,
    verbose=True
)

In [122]:
response = agent.invoke({
    "messages":[
        {
            "role":"user",
            "content":"What would be a good demo to show"
        }
    ]
})
for message in response["messages"]:
    print(type(message).__name__)
    print(message.content)
    print("------")

HumanMessage
What would be a good demo to show
------
AIMessage
To give you the best recommendation, I need to know a little more about your **audience** and your **goal**. A demo for a CEO is very different from a demo for a group of software engineers.

However, I can categorize the "best" demos based on common scenarios. Choose the category that fits your situation:

### 1. If you are a Software Developer (Technical Demo)
*The goal here is to show technical competence, problem-solving, and code quality.*
*   **The "Real-World Problem" Demo:** Don't just show code; show a problem. "Here is a slow database query; here is how I implemented caching to make it 10x faster."
*   **The API/Integration Demo:** Show how your code talks to other systems (e.g., "Our app automatically pushes data to Slack when a sale happens").
*   **The "Under the Hood" Demo:** If you are applying for a backend role, show a complex algorithm or a system architecture diagram alongside the working code.

### 2. I

In [ ]:
memory.load_memory_variables({})

Reminder: Download your notebook to you local computer to save your work.